# Attack FOA — v1

Bật Internet; bật GPU cho attack/caption local. Source code được clone/pull từ GitHub vào /kaggle/working, không cần upload code thành Kaggle Dataset.
Chỉ đổi CONFIG_NAME để chọn model/thực nghiệm; chỉnh đường dẫn dữ liệu một lần.
Secrets: OPENAI_API_KEY cho GPT/judge, GEMINI_API_KEY cho Gemini. Không ghi key vào notebook.
Mọi xử lý nằm trong các file Python hiện có. Xem src/FOA-Attack/config/README.md.
Mặc định chạy smoke (không phải baseline); đổi sang attack_foa_baseline_v1 cho thí nghiệm paper.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/zintomvn/Code-Adversarial-Attack-on-VLM.git"
GIT_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/Code-Adversarial-Attack-on-VLM")
CONFIG_NAME = "attack_foa_smoke_v1"
OUTPUT_DIR = "/kaggle/working/foa_outputs"
SOURCE_DIR = "/kaggle/input/foa-images/source"
TARGET_DIR = "/kaggle/input/foa-images/target"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    assert (REPO_DIR / ".git").exists(), f"Không phải Git repository: {REPO_DIR}"
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)

RUNNER = REPO_DIR / "src" / "FOA-Attack" / "utils.py"
assert RUNNER.is_file(), f"Chỉnh REPO_DIR: không tìm thấy {RUNNER}"


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
                "hydra-core==1.3.2", "omegaconf==2.3.0", "PyYAML==6.0.2", "wandb==0.19.1"], check=True)
subprocess.run([sys.executable, str(RUNNER), "setup", "--config-name", CONFIG_NAME], check=True)


In [ ]:
command = [sys.executable, str(RUNNER), "run", "--config-name", CONFIG_NAME, "--output", OUTPUT_DIR]
command += ["--source", SOURCE_DIR, "--target", TARGET_DIR]
subprocess.run(command, check=True)


In [ ]:
subprocess.run([sys.executable, str(RUNNER), "export", "--config-name", CONFIG_NAME, "--output", OUTPUT_DIR], check=True)
from IPython.display import FileLink, display
for archive in sorted(Path(OUTPUT_DIR).glob(f"{CONFIG_NAME}_*.zip")):
    display(FileLink(str(archive)))
